# 实践项目 05：空间转录组表达超分辨率

本 Notebook 使用课程准备的配对 H&E、LR、HR 和 split 数据。LR 表示 16 μm Snap25 输入，HR 表示 2 μm 参考表达图，模型学习在细网格上估计一个高表达基因的局部表达。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码单元格保留行末注释，说明每一步的输入、处理和输出；参考实现与实践顺序对应。

## 任务总览

1. 核对四个字段的 shape、split 数量和 Snap25 表达范围。
2. 选择中心区域，查看同一位置的大图、小图、H&E、LR 和 HR。
3. 补全 H&E 与粗尺度表达联合输入的轻量网络。
4. 完成 log 空间损失和 8×8 区域总量约束。
5. 比较模型与插值基线的 MAE、相关性、聚合误差和空间图。

## 需要保存的结果

`task5_data_visualization.png`、`task5_scale_overview.png`、`task5_training_curve.png`、`task5_prediction_visualization.png`、`task5_result.json`。


In [ ]:
from pathlib import Path  # 导入当前步骤需要的工具
import json, random  # 导入当前步骤需要的工具
import numpy as np  # 导入当前步骤需要的工具
import matplotlib.pyplot as plt  # 导入当前步骤需要的工具
import torch  # 导入当前步骤需要的工具
import torch.nn as nn  # 导入当前步骤需要的工具
from torch.utils.data import Dataset, DataLoader  # 导入当前步骤需要的工具

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 固定随机状态以便复现实验
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 保存当前步骤使用的中间结果
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)  # 保存输出文件目录
DATA_PATH = None  # 可在本地填写课程 NPZ 路径；Kaggle 自动查找固定文件名
candidates = sorted(Path('/kaggle/input').rglob('kydw-try-a05-paired-patches.npz'))  # 读取本任务需要的数据
if DATA_PATH is None and candidates: DATA_PATH = candidates[0]  # 根据当前条件选择处理分支
assert DATA_PATH is not None, '请挂载包含 kydw-try-a05-paired-patches.npz 的课程数据集。'  # 执行当前步骤并保留结果
data = np.load(DATA_PATH, allow_pickle=True)  # 读取本任务需要的数据
required = {'he','lr','hr','split'}  # 核对输入字段
assert required.issubset(data.files), required - set(data.files)  # 执行当前步骤并保留结果
he = data['he'].astype(np.float32) / 255.0  # 读取 H&E 图像并归一化
lr = data['lr'].astype(np.float32)  # 读取 16 μm 粗尺度表达总量
hr = data['hr'].astype(np.float32)  # 读取 2 μm 参考表达图
split = data['split'].astype(str)  # 读取预先划分的数据集合
print('he/lr/hr:', he.shape, lr.shape, hr.shape, 'splits:', {k: int((split == k).sum()) for k in np.unique(split)})  # 显示核对结果


## 任务 1：核对输入字段与空间划分

H&E、LR 和 HR 覆盖同一空间区域。LR 在每个 8×8 区域内记录粗尺度总量；`split` 是课程数据预先提供的空间划分。


In [ ]:
summary = {
    'shapes': {'he': list(he.shape), 'lr': list(lr.shape), 'hr': list(hr.shape)},
    'split_counts': {kind: int((split == kind).sum()) for kind in ('train', 'validation', 'test')},
    'lr_nonzero_ratio': float((lr > 0).mean()),
    'hr_nonzero_ratio': float((hr > 0).mean()),
    'lr_max': float(lr.max()), 'hr_max': float(hr.max()),
}
assert set(np.unique(split)) <= {'train', 'validation', 'test'}
print(summary)


## 任务 2：配对大图、小图与表达图

从 train 样本中选择组织覆盖较完整、表达信号可见的一项，把 H&E、LR 粗尺度密度和 HR 参考表达图放在同一行，再截取中心区域放大。三列来自同一个空间区域，色标可以分别设置。


In [ ]:
train_ids = np.where(split == 'train')[0]
coverage_score = (he[train_ids].mean(axis=1) < .98).mean(axis=(1, 2)) + 2 * (hr[train_ids] > 0).mean(axis=(1, 2))
sample_index = int(train_ids[np.argmax(coverage_score)])
lr_density = lr[sample_index, 0] / 64.0; hr_reference = hr[sample_index, 0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for axis, value, title, cmap in zip(axes, [np.moveaxis(he[sample_index], 0, -1), lr_density, hr_reference], ['H&E', '16 μm LR Snap25', '2 μm HR Snap25'], [None, 'magma', 'magma']):
    axis.imshow(value, cmap=cmap); axis.set_title(title); axis.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'task5_data_visualization.png', dpi=180); plt.close(fig)
crop = (slice(48, 208), slice(48, 208))
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for column, value in enumerate([np.moveaxis(he[sample_index], 0, -1), lr_density, hr_reference]):
    axes[0, column].imshow(value, cmap=None if column == 0 else 'magma'); axes[1, column].imshow(value[crop], cmap=None if column == 0 else 'magma')
    axes[0, column].set_title('full 256×256'); axes[1, column].set_title('center 160×160'); axes[0, column].axis('off'); axes[1, column].axis('off')
fig.tight_layout(); fig.savefig(OUT / 'task5_scale_overview.png', dpi=180); plt.close(fig)
print('sample index:', sample_index)


## 任务 3：融合网络

输入为 4 个通道（H&E 三通道和 LR 密度），输出为 1 个通道的细尺度表达预测。


In [ ]:
def aggregate8(x):
    return torch.nn.functional.avg_pool2d(x, 8, 8) * 64

def loss_fn(pred_log, target_log, lr_raw):
    pred = torch.expm1(pred_log).clamp_min(0); target = torch.expm1(target_log).clamp_min(0)
    l1 = (pred_log - target_log).abs().mean()
    observed = torch.nn.functional.avg_pool2d(lr_raw, 8, 8) * 64
    consistency = (aggregate8(pred) - observed).abs().mean()
    return l1 + .1 * consistency


## 任务 4：损失和 8×8 总量约束

预测值按 8×8 区域求和后，应与 LR 中的粗尺度总量接近；这项约束让输出保留观测到的总量。


In [ ]:
train_loader = DataLoader(STDataset('train'), batch_size=4, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(60):
    model.train()
    for x, target_log, lr_raw, _ in train_loader:
        x, target_log, lr_raw = x.to(DEVICE), target_log.to(DEVICE), lr_raw.to(DEVICE)
        optimizer.zero_grad(); prediction = model(x)
        loss = loss_fn(torch.log1p(prediction), target_log, lr_raw); loss.backward(); optimizer.step()
model.eval()
test_ids = np.where(split == 'test')[0]
with torch.no_grad():
    x = torch.from_numpy(np.concatenate([he[test_ids], np.log1p(lr[test_ids] / 64.0)], axis=1)).to(DEVICE)
    prediction = model(x).cpu().numpy()[:, 0]
target = hr[test_ids, 0]; interpolation = lr[test_ids, 0] / 64.0
result = {
    'gene': 'Snap25',
    'test_mae_model': float(np.abs(prediction - target).mean()),
    'test_mae_interpolation': float(np.abs(interpolation - target).mean()),
    'test_pearson_model': float(np.corrcoef(prediction.ravel(), target.ravel())[0, 1]),
}
(OUT / 'task5_result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(result)


## 任务 5：训练与结果比较

完成训练后，比较模型与插值基线的 MAE、Pearson 相关性和 8×8 聚合误差，并查看预测、参考和误差图。


In [ ]:
# ===== 项目05·任务5·参考实现（开始） =====
# 参考实现：完成训练、验证选模、模型与插值基线比较，以及结果保存。
# 参考答案会使用较长训练和稳定的模型设置，但运行后请以自己的输出为准。
# ===== 项目05·任务5·参考实现（结束） =====
print('输出文件应写入', OUT)  # 显示便于检查的关键信息
